In [2]:
#imports 
%load_ext autoreload
%autoreload 2
import matplotlib.pyplot as plt
import pandas as pd
from utils import clean_df, build_regex


In [3]:
llama_bai_path = '/projectnb/ivc-ml/micahb/lm-mental-health-eval/results/meta-llama__Llama-3.1-8B-Instruct/final/samples_bai_2026-03-08T16-16-03.530976.jsonl'
llama_bdi_path = '/projectnb/ivc-ml/micahb/lm-mental-health-eval/results/meta-llama__Llama-3.1-8B-Instruct/final/samples_bdi_2026-03-08T16-16-33.615709.jsonl'

gemma_bai_path = '/projectnb/ivc-ml/micahb/lm-mental-health-eval/results/google__gemma-3-12b-it/final/samples_bai_2026-03-08T17-15-57.804624.jsonl'
gemma_bdi_path = '/projectnb/ivc-ml/micahb/lm-mental-health-eval/results/google__gemma-3-12b-it/final/samples_bdi_2026-03-08T17-23-06.945392.jsonl'

patterns = build_regex()
llama_bdi_df = clean_df(pd.read_json(llama_bdi_path, lines=True), patterns)
llama_bai_df = clean_df(pd.read_json(llama_bai_path, lines=True), patterns)

gemma_bdi_df = clean_df(pd.read_json(gemma_bdi_path, lines=True), patterns)
gemma_bai_df = clean_df(pd.read_json(gemma_bai_path, lines=True), patterns)



In [4]:
import markdown


def get_sample(df): 
    df['statement_id'] = df['prompt_id'].apply(lambda x: x.split('_r')[0])
    reshape = df[['statement_id', 'response', 'round']].pivot(index='statement_id', columns='round', values=['response'])
    sample = reshape.apply(lambda row: row.sample(random_state=234).values[0], axis=1)
    return sample

llama_df = pd.concat([llama_bai_df, llama_bdi_df], axis=0)
gemma_df = pd.concat([gemma_bai_df, gemma_bdi_df], axis=0)

llama_sample = get_sample(llama_df).apply(lambda x: markdown.markdown(x))
gemma_sample = get_sample(gemma_df).apply(lambda x: markdown.markdown(x))

In [5]:
gemma_sample.to_csv('./gemma_eval.csv')
llama_sample.to_csv('./llama_eval.csv')